In [1]:
# -*- coding: utf-8 -*-
"""
Text understanding inference script
"""
import os
import json
import argparse
from PIL import Image
import torch
import time
from transformers import AutoConfig, AutoTokenizer
import sys
sys.path.append("/mnt/data1/jiwon/Lumina-DiMOO")

from config import SPECIAL_TOKENS
from model import LLaDAForMultiModalGeneration
from utils.image_utils import preprocess_image, encode_img_with_breaks, calculate_vq_params, generate_crop_size_list, var_center_crop, add_break_line, encode_img_with_breaks_fixed
from generators.text_understanding_generator import generate_text_understanding
from utils.prompt_utils import generate_multimodal_understanding_prompt
from utils.generation_utils import setup_seed
setup_seed(42)



# Special tokens
MASK = SPECIAL_TOKENS["mask_token"]
NEW_LINE = SPECIAL_TOKENS["newline_token"]
BOA = SPECIAL_TOKENS["answer_start"]  # Begin of Answer
EOA = SPECIAL_TOKENS["answer_end"]    # End of Answer
BOI = SPECIAL_TOKENS["boi"]           # Begin of Image
EOI = SPECIAL_TOKENS["eoi"]           # End of Image

# Load model and tokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained('Alpha-VLLM/Lumina-DiMOO', trust_remote_code=True)
# if os.path.exists(args.checkpoint):
#     print("Loading model from local checkpoint: ", args.checkpoint)
#     base_config = AutoConfig.from_pretrained(args.checkpoint, trust_remote_code=True)
#     # Load model with correct config
#     model = LLaDAForMultiModalGeneration(base_config)
# else:
model = LLaDAForMultiModalGeneration.from_pretrained(
    'Alpha-VLLM/Lumina-DiMOO', 
    torch_dtype=torch.bfloat16, 
    flash_attention=True,
)
    

`torch_dtype` is deprecated! Use `dtype` instead!


Initializing MMadaModelLM with config: LLaDAConfig {
  "activation_type": "silu",
  "alibi": false,
  "alibi_bias_max": 8.0,
  "architectures": [
    "LLaDAForMultiModalGeneration"
  ],
  "attention_dropout": 0.0,
  "attention_layer_norm": false,
  "attention_layer_norm_with_affine": true,
  "auto_map": {
    "AutoConfig": "configuration_llada.LLaDAConfig",
    "AutoModel": "modeling_llada.LLaDAModelLM",
    "AutoModelForCausalLM": "modeling_llada.LLaDAModelLM"
  },
  "bias_for_layer_norm": false,
  "block_group_size": 1,
  "block_type": "llama",
  "d_model": 4096,
  "dtype": "bfloat16",
  "embedding_dropout": 0.0,
  "embedding_size": 134548,
  "eos_token_id": 126081,
  "flash_attention": true,
  "include_bias": false,
  "include_qkv_bias": false,
  "init_cutoff_factor": null,
  "init_device": "meta",
  "init_fn": "mitchell",
  "init_std": 0.02,
  "input_emb_norm": false,
  "layer_norm_type": "rms",
  "layer_norm_with_affine": true,
  "mask_token_id": 126336,
  "max_sequence_length": 4

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
model

LLaDAForMultiModalGeneration(
  (model): LLaDAModel(
    (transformer): ModuleDict(
      (wte): Embedding(134548, 4096)
      (emb_drop): Dropout(p=0.0, inplace=False)
      (ln_f): RMSLayerNorm()
      (blocks): ModuleList(
        (0-31): 32 x LLaDALlamaBlock(
          (dropout): Dropout(p=0.0, inplace=False)
          (act): SiLU()
          (attn_out): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_out): Linear(in_features=12288, out_features=4096, bias=False)
          (rotary_emb): RotaryEmbedding()
          (attn_norm): RMSLayerNorm()
          (ff_norm): RMSLayerNorm()
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
      